# Train/test split and missing values

**Aim:** split the cleaned loans by issue year, then fill the remaining missing values using the training years only.

Everything before this point is in `src/credit_risk/data_prep.py` (notebook 01). This notebook loads through that file and does not repeat the exploration.

**Sections**
1. Loading the data
2. Train/test split
3. Missing values

In [1]:
import sys
sys.path.insert(0, "../src")

%load_ext autoreload
%autoreload 2

import pandas as pd
import credit_risk.data_prep as dp

pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 250)

DATA_PATH = "../data/raw/accepted_2007_to_2018Q4.csv.gz"

## 1. Loading the data

Load only the columns we use, label the loans, keep the usable issue years, and convert the text columns.

In [2]:
df = dp.load_needed(DATA_PATH)
labelled = dp.build_resolved(df)
data = dp.restrict_vintages(labelled)
data = dp.clean_types(data)

# final data: 676,302 labelled loans, 41 columns 
# = id, loan_status, issue_d, issue_year, default, 5 LGD columns (not model inputs), 31 features
print(data.shape)

(676302, 41)


## 2. Train/test split

**Question:** which loans do we fit on, and which do we test on?

In [3]:
# train on 2007-2014, test on 2015
train = data[data["issue_year"] <= 2014]
test = data[data["issue_year"] == 2015]

In [4]:
for name, part in [("train", train), ("test", test)]:
    print(name, len(part))
    print(part.groupby("term")["default"].agg(["count", "mean"]).round(3), "\n")

train 393276
       count   mean
term               
36    337996  0.132
60     55280  0.252 

test 283026
       count   mean
term               
36    283026  0.149 



**Decision**
Train on issue years 2007-2014 (393,276 loans). Test on 2015 (283,026 loans).

**Why**
- 2015 is the last year in which 36-month loans are fully labelled (notebook 01, section 3).
- Testing on a later year than training mimics real use: the model is fitted on past loans and scores newer ones. A random split would let it learn from loans issued after some of the ones it is tested on.
- Default rates rise over time. For 36-month loans it goes from 13.2% in train to 14.9% in test.

**Limitation**
The test set is 36-month only. All 55,280 60-month loans are in train, so the model can't be tested on 60-month loans.

## 3. Missing values

**Question:** which columns still have missing values, and how do the train and test years compare?

In [5]:
# % missing in train and test, for the model columns that have any
feats = dp.MODEL_FEATURES
miss = pd.DataFrame({
    "train %": train[feats].isnull().mean().mul(100),
    "test %": test[feats].isnull().mean().mul(100),
}).round(2)

miss[(miss > 0).any(axis=1)]

,train %,test %
emp_length,4.74,6.70
delinq_2yrs,0.01,0.00
credit_history_years,0.01,0.00
inq_last_6mths,0.01,0.00
mths_since_last_delinq,54.42,47.99
open_acc,0.01,0.00
pub_rec,0.01,0.00
revol_util,0.07,0.04
total_acc,0.01,0.00
collections_12_mths_ex_med,0.04,0.00


In [6]:
col = train["mths_since_last_delinq"]

# spread of the values that are filled in
print(col.describe().round(1))

# share of very long gaps
print("over 120 months:", (col > 120).mean().round(4))

# default rate: never delinquent (missing) vs delinquent at some point
print(train.groupby(col.isnull())["default"].agg(["count", "mean"]).round(3))

# default rate by time since the last delinquency, with missing as its own group
bins = pd.cut(col, [0, 6, 12, 24, 36, 60, 120, col.max()], include_lowest=True)
train.groupby(bins.cat.add_categories("missing").fillna("missing"), observed=True)["default"].agg(["count", "mean"]).round(3)

count    179247.0
mean         34.3
std          21.8
min           0.0
25%          16.0
50%          31.0
75%          50.0
max         188.0
Name: mths_since_last_delinq, dtype: float64
over 120 months: 0.0001
                         count   mean
mths_since_last_delinq               
False                   179247  0.151
True                    214029  0.146


,count,mean
mths_since_last_delinq,,
"(-0.001, 6.0]",13071,0.166
"(6.0, 12.0]",20476,0.157
"(12.0, 24.0]",37714,0.150
"(24.0, 36.0]",32298,0.146
"(36.0, 60.0]",47772,0.147
"(60.0, 120.0]",27882,0.155
"(120.0, 188.0]",34,0.176
missing,214029,0.146


**Decision**
Leave missing values as NaN. In the WoE notebook each feature gets a separate bin for missing values, with its own WoE, so nothing needs filling here.

**Why**
- `mths_since_last_delinq` (54% missing in train, 48% in test): missing means never delinquent. Those loans default at 14.6%, against 15.1% for loans with a delinquency on record, and the rate by time since delinquency is not monotonic. A bin of its own captures this without choosing a fill value.
- `emp_length` (4.7% in train, 6.7% in test): same treatment.
- The other 13 columns are 0.35% missing or less in train and 0.04% or less in test, almost all from 2007-2008. The WoE notebook decides whether those tiny missing groups get their own bin or are merged with the bin of the closest default rate.
- If a later model uses raw values instead of WoE, it fills with the train median and adds a missing flag, fitted on train only.